# Traffic DRL Orchestration Demo

This notebook demonstrates how to use the centralized configuration and run ID management systems
for training and evaluation orchestration in the traffic DRL project.

**Note**: This is a demonstration notebook showing the orchestration flow. Actual implementations
of the functions would be needed to make this runnable.


## 1. Import Required Modules

First, we import all the necessary modules for configuration management, run ID handling,
and the core training/evaluation functions.

In [ ]:
# Import configuration management
from traffic_drl.config import load_env_config, load_train_config, load_eval_config

# Import run ID management
from traffic_drl.run_id import (
    get_default_run_id,
    ensure_run_directories,
    create_sumo_output_options,
    get_tripinfo_path,
    get_metrics_path
)

# Import core functions (these would be implemented in the actual codebase)
from traffic_drl.environment.make_env import (
    create_sumo_env,
    make_vectorized_environment
)
from traffic_drl.environment.wrappers import wrap_environment
from traffic_drl.train.train_dqn import (
    build_dqn_model,
    train_dqn,
    run_dqn_pilot
)
from traffic_drl.train.train_ppo import (
    build_ppo_model,
    train_ppo
)
from traffic_drl.evaluation.evaluate_benchmark import run_benchmark

# Import utilities
import yaml
from pathlib import Path
import torch
import numpy as np


## 2. Load Configurations

We'll load our training and evaluation configurations using the centralized config loader.

In [ ]:
# Load training configuration
train_config = load_train_config("configs/train/dqn_phase1.yaml")
print("Training config loaded:")
print(f"  Experiment: {train_config['experiment']['name']}")
print(f"  Seed: {train_config['experiment']['seed']}")
print(f"  Device: {train_config['experiment']['device']}")
print()

# Load evaluation configuration
eval_config = load_eval_config("configs/evaluation/phase1_validation.yaml")
print("Evaluation config loaded:")
print(f"  Benchmark: {eval_config['benchmark']['name']}")
print(f"  Split: {eval_config['benchmark']['split']}")
print(f"  Deterministic: {eval_config['benchmark']['deterministic']}")
print()

# Optionally, we could also load a specific environment config
# env_config = load_env_config(train_config['environment']['env_config_path'])


## 3. Generate Run ID and Setup Directories

Now we generate a unique run ID for this experiment and set up the necessary directories.

In [ ]:
# Generate a run ID for this experiment
run_id = get_default_run_id(prefix="dqn_phase1")
print(f"Generated run ID: {run_id}")

# Ensure the necessary directories exist
tripinfo_dir, results_dir = ensure_run_directories(run_id)
print(f"Tripinfo directory: {tripinfo_dir}")
print(f"Results directory: {results_dir}")

# Get SUMO output options for this run
sumo_options = create_sumo_output_options(run_id)
print("SUMO output options:")
for key, value in sumo_options.items():
    print(f"  {key}: {value}")

# Get specific file paths we might need
tripinfo_path = get_tripinfo_path(run_id)
metrics_path = get_metrics_path(run_id)
print(f"Tripinfo path: {tripinfo_path}")
print(f"Metrics path: {metrics_path}")


## 4. Create Environment

Next, we create the SUMO-RL environment using our configuration and run ID.

In [ ]:
# Create the base SUMO-RL environment
# Note: In practice, we would need a specific route file for this
# For demonstration, we'll use a placeholder
route_file = "scenarios/train/tr_01.rou.xml"  # Example route file

# Create base environment
base_env = create_sumo_env(
    config=train_config['environment'],  # Pass the environment section of config
    route_file=route_file,
    fixed_ts=False,  # We want a learning agent, not fixed-time
    seed=train_config['experiment']['seed'],
    use_gui=False
)
print(f"Base environment created: {type(base_env)}")

# Wrap the environment with our standard wrappers
# We'll use the default wrapping behavior (MultiScenarioWrapper if needed, etc.)
wrapped_env = wrap_environment(
    env=base_env,
    scenario_sampler=None,  # We'll handle scenario sampling elsewhere for simplicity
)
print(f"Wrapped environment: {type(wrapped_env)}")

# Optionally, we could apply observation normalization
# For now, we'll skip VecNormalize for simplicity in this demo


## 5. Build and Train Model

Now we'll build our DQN model and train it using our configuration.

In [ ]:
# Build the DQN model
# Note: build_dqn_model expects the model_hyperparameters section of the config
model = build_dqn_model(
    env=wrapped_env,
    config=train_config['model_hyperparameters'],  # Just the hyperparameters section
    seed=train_config['experiment']['seed'],
    tensorboard_log=str(results_dir / "tensorboard_logs")
)
print(f"DQN model built: {type(model)}")

# Train the model
# We'll use the run_dqn_pilot function which handles the full training loop
# This function now accepts either a config dict or a path to a config file
trained_model = run_dqn_pilot(
    config=train_config,  # Pass the full config - it will extract what it needs
    checkpoint_dir=str(results_dir / "checkpoints"),
    total_timesteps=train_config['training_control']['total_timesteps'],
    seed=train_config['experiment']['seed']
)
print(f"Model training completed: {type(trained_model)}")

# Save the final model
model_save_path = results_dir / "final_model.zip"
trained_model.save(str(model_save_path))
print(f"Final model saved to: {model_save_path}")


## 6. Run Evaluation/Benchmark

Finally, we'll run the benchmark evaluation to compare our trained model against baselines.

In [ ]:
# Update the evaluation config with our trained model path
# In a real scenario, we might load the eval config and modify the artifacts section
eval_config_updated = eval_config.copy()
eval_config_updated['artifacts']['model_checkpoint'] = str(model_save_path)
eval_config_updated['artifacts']['vec_normalize_stats'] = str(results_dir / "checkpoints" / "vec_normalize.pkl")

# Run the benchmark evaluation
# The run_benchmark function now accepts either a config dict or a path to a config file
benchmark_report = run_benchmark(
    config=eval_config_updated,  # Pass the updated config
)
print(f"Benchmark evaluation completed: {type(benchmark_report)}")

# Save the benchmark results
results_save_path = results_dir / "benchmark_report.json"
# In practice, we would serialize the BenchmarkReport object
# For this demo, we'll just show what we would do
print(f"Benchmark results would be saved to: {results_save_path}")
print("\nBenchmark report summary:")
print(f"  Baseline controller: {benchmark_report.baseline_controller}")
print(f"  Number of scenario summaries: {len(benchmark_report.summaries)}")


## 7. Alternative: Using Pre-loaded Configs

We can also demonstrate how to use pre-loaded configs with functions that accept file paths.

In [ ]:
# Example: Using config file paths directly with functions that support it
# (Assuming the functions have been updated to accept path strings)

# Generate a new run ID for this alternative approach
run_id_alt = get_default_run_id(prefix="dqn_alt")
print(f"Alternative run ID: {run_id_alt}")

# Ensure directories exist
tripinfo_dir_alt, results_dir_alt = ensure_run_directories(run_id_alt)

# Create environment using config path directly
# Note: This would require the make_env functions to accept config paths
# base_env_alt = create_sumo_env(
#     config=train_config['environment']['env_config_path'],  # Pass path directly
#     route_file=route_file,
#     fixed_ts=False,
#     seed=train_config['experiment']['seed'],
#     use_gui=False
# )

# Or train using config path directly
# Note: This would require run_dqn_pilot to accept config paths
# trained_model_alt = run_dqn_pilot(
#     config="configs/train/dqn_phase1.yaml",  # Pass path directly
#     checkpoint_dir=str(results_dir_alt / "checkpoints"),
#     total_timesteps=train_config['training_control']['total_timesteps'],
#     seed=train_config['experiment']['seed']
# )

# Run evaluation using config path directly
# benchmark_report_alt = run_benchmark(
#     config="configs/evaluation/phase1_validation.yaml",  # Pass path directly
# )

print("Alternative approach using config paths directly would work if functions support it.")
print("This provides flexibility in how configs are passed to functions.")
   

## 8. Summary

This demo shows how the centralized configuration and run ID management systems
work together to create a clean, organized orchestration workflow:

1. **Centralized Config Loading**: All configs loaded through `load_*_config()` functions
2. **Run ID Management**: Unique identifiers generated and directory structure created automatically
3. **Flexible Function Interfaces**: Functions accept either config dictionaries or file paths
4. **Automatic Path Generation**: SUMO options and file paths generated from run ID
5. **Clean Organization**: All outputs organized under `outputs/<type>/<run_id>/`

### Benefits:
- Eliminates scattered config loading code
- Ensures consistent experiment identification
- Provides clear separation of concerns
- Makes it easy to reproduce experiments
- Reduces boilerplate code in training/evaluation scripts

### Next Steps for Implementation:
1. Implement the actual function bodies (currently raise NotImplementedError)
2. Update functions to accept config paths as demonstrated
3. Integrate run ID passing throughout the orchestration pipeline
4. Add proper error handling and validation
5. Create actual route files and SUMO configurations for testing
